# ELECTRA + Static Features + DANN

Self-contained Colab notebook for the final experiment.

Train on judge labels from:
- Shrishti `train.csv`
- OneStop `ood_onestop.csv`
- RACE middle/high `ood_race-*.csv`
- CNN/DailyMail `cnn_dailymail_2k.csv`

Validate on Shrishti `val.csv`.

OOD test on:
- XSum
- CoQA
- WeeBit
- CommonLit

Model: ELECTRA-large pooled embedding + scaled static readability features, with DANN over the fused representation.

In [ ]:
!pip install -q transformers scikit-learn torch pandas matplotlib seaborn tqdm textstat joblib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Config
DRIVE_SHRISHTI_CLEAN = "/content/drive/MyDrive/BeyondFK/clean_dataset"
DRIVE_MULTI_CLEAN = "/content/drive/MyDrive/BeyondFK/trail/judge_multi_corpus/clean_dataset"
DRIVE_OUT_DIR = "/content/drive/MyDrive/BeyondFK/trail/electra_static_cnn_ose_race_multi_ood_dann"

MODEL_NAME = "google/electra-large-discriminator"
TEXT_COL = "full_text"
LABEL_COL = "education_level_judge"
SOURCE_COL = "source_dataset"

MAX_LEN = 512
BATCH_SIZE = 4
RNG_SEED = 42
LABEL_SMOOTHING = 0.1
DEDUPE_TEXT = True
EVAL_LABELS = [0, 1, 2]

# DANN
GRL_LAMBDA_MAX = 0.15
DOMAIN_LOSS_ALPHA = 0.1
PHASE1_EPOCHS = 2
PHASE1_LR = 1e-3
PHASE2_EPOCHS = 1
PHASE2_LR = 2e-5
PARTIAL_FREEZE_LAYERS = 8
WARMUP_STEPS = 100

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

TRAIN_SOURCES = [
    (f"{DRIVE_SHRISHTI_CLEAN}/train.csv", "shrishti_train"),
    (f"{DRIVE_SHRISHTI_CLEAN}/ood_onestop.csv", "onestop_train"),
    (f"{DRIVE_SHRISHTI_CLEAN}/ood_race-middle.csv", "race_middle_train"),
    (f"{DRIVE_SHRISHTI_CLEAN}/ood_race-high.csv", "race_high_train"),
    (f"{DRIVE_MULTI_CLEAN}/cnn_dailymail_2k.csv", "cnn_dailymail_train"),
]

OOD_SOURCES = [
    (f"{DRIVE_MULTI_CLEAN}/xsum_2k.csv", "xsum_ood"),
    (f"{DRIVE_MULTI_CLEAN}/coqa_2k.csv", "coqa_ood"),
    (f"{DRIVE_MULTI_CLEAN}/weebit_500.csv", "weebit_ood"),
    (f"{DRIVE_MULTI_CLEAN}/commonlit_500.csv", "commonlit_ood"),
]

# Adversarial concept set from the repo/paper artifacts. The loader below uses the
# first path that exists, so Colab Drive and local runs both work.
ADV_CONCEPT_PATH_CANDIDATES = [
    "/content/drive/MyDrive/BeyondFK/Final_Architecture-Pillar_B plus/data/adv_concept.csv",
    "/content/drive/MyDrive/BeyondFK/adv_concept.csv",
    "../Final_Architecture-Pillar_B plus/data/adv_concept.csv",
    "Final_Architecture-Pillar_B plus/data/adv_concept.csv",
]

In [ ]:
# Imports and setup
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import re
from collections import Counter
from math import log2
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import textstat
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from torch.nn import Parameter, ParameterList
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
CM_DIR = os.path.join(DRIVE_OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)
print("Device:", device)
print("OUT:", DRIVE_OUT_DIR)

In [ ]:
# Static features, adapted from Shrishti step3d_static_features.py
READABILITY = [
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "coleman_liau_index", "automated_readability_index",
    "dale_chall_readability_score", "difficult_words",
    "linsear_write_formula", "gunning_fog", "text_standard",
    "spache_readability", "mcalpine_eflaw",
]

WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")


def _safe(fn, *args, **kwargs):
    try:
        v = fn(*args, **kwargs)
        if isinstance(v, str):
            m = re.search(r"\d+", v)
            return float(m.group(0)) if m else float("nan")
        return float(v)
    except Exception:
        return float("nan")


def _word_entropy(words):
    if not words:
        return 0.0
    c = Counter(w.lower() for w in words)
    n = sum(c.values())
    return -sum((v / n) * log2(v / n) for v in c.values())


def static_features(text):
    text = (text or "").strip()
    feats = {}
    for name in READABILITY:
        feats[f"stat_{name}"] = _safe(getattr(textstat, name), text)

    words = WORD_RE.findall(text)
    sentences = [s for s in SENT_SPLIT.split(text) if s.strip()]
    n_words = len(words)
    n_sents = max(len(sentences), 1)

    feats["stat_char_count"] = len(text)
    feats["stat_word_count"] = n_words
    feats["stat_unique_words"] = len(set(w.lower() for w in words))
    feats["stat_ttr"] = feats["stat_unique_words"] / n_words if n_words else 0.0
    feats["stat_avg_word_len"] = float(np.mean([len(w) for w in words])) if words else 0.0
    feats["stat_long_word_ratio"] = sum(1 for w in words if len(w) > 6) / n_words if n_words else 0.0
    feats["stat_sentence_count"] = len(sentences)
    feats["stat_avg_sent_len"] = n_words / n_sents
    feats["stat_max_sent_len"] = max((len(WORD_RE.findall(s)) for s in sentences), default=0)
    feats["stat_word_entropy"] = _word_entropy(words)
    feats["stat_n_question"] = text.count("?")
    feats["stat_n_exclaim"] = text.count("!")
    feats["stat_n_comma"] = text.count(",")
    feats["stat_n_semicolon"] = text.count(";")
    feats["stat_n_paren"] = text.count("(") + text.count(")")
    feats["stat_n_quote"] = text.count('"')
    feats["stat_n_paragraphs"] = text.count("\n\n") + 1
    feats["stat_n_newlines"] = text.count("\n")

    lower = text.lower()
    feats["stat_has_because"] = float(" because" in lower)
    feats["stat_has_therefore"] = float("therefore" in lower)
    feats["stat_has_example"] = float("example" in lower or "e.g." in lower)
    feats["stat_has_multiple_choice"] = float(any(m in lower for m in [" (a)", " (b)", " (c)"]))
    return feats


def compute_static_matrix(df, feature_names=None, desc="static"):
    rows = [static_features(t) for t in tqdm(df[TEXT_COL].astype(str).tolist(), desc=desc)]
    sdf = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if feature_names is None:
        feature_names = sorted(c for c in sdf.columns if c.startswith("stat_"))
    for c in feature_names:
        if c not in sdf.columns:
            sdf[c] = 0.0
    return sdf[feature_names].astype("float32").values, feature_names

print("Static feature functions loaded")

In [ ]:
# Load data and compute scaled static features
def _text_key(text: str) -> str:
    return hashlib.sha256(str(text).strip().encode("utf-8")).hexdigest()


def load_clean_csv(path: str, pool: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(p)
    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"{path}: need {TEXT_COL} and {LABEL_COL}")
    out = pd.DataFrame()
    out[TEXT_COL] = df[TEXT_COL].astype(str)
    out["label_str"] = df[LABEL_COL].astype(str).str.strip().str.lower()
    out["pool"] = pool
    out[SOURCE_COL] = df[SOURCE_COL].astype(str) if SOURCE_COL in df.columns else pool
    bad = ~out["label_str"].isin(label2id)
    if bad.any():
        print(f"[{pool}] dropping {bad.sum()} bad labels")
        out = out[~bad]
    out["label_id"] = out["label_str"].map(label2id).astype(int)
    return out.reset_index(drop=True)


def dedupe_train(df: pd.DataFrame) -> pd.DataFrame:
    if not DEDUPE_TEXT:
        return df
    priority = [p for _, p in TRAIN_SOURCES]
    rank = {p: i for i, p in enumerate(priority)}
    out = df.copy()
    out["_rk"] = out["pool"].map(lambda x: rank.get(x, 999))
    out["_key"] = out[TEXT_COL].map(_text_key)
    out = out.sort_values("_rk").drop_duplicates("_key", keep="first")
    return out.drop(columns=["_rk", "_key"]).reset_index(drop=True)


def first_existing_path(paths: List[str]) -> str:
    for path in paths:
        if Path(path).exists():
            return path
    raise FileNotFoundError("None of these adv_concept paths exist: " + ", ".join(paths))


def load_adv_concept_csv(paths: List[str], pool: str = "adv_concept_ood") -> pd.DataFrame:
    path = first_existing_path(paths)
    df = pd.read_csv(path)
    required = {"text", "true_level"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")

    out = pd.DataFrame()
    out[TEXT_COL] = df["text"].astype(str)
    out["label_str"] = df["true_level"].astype(str).str.strip().str.lower()
    out["pool"] = pool
    out[SOURCE_COL] = df["source"].astype(str) if "source" in df.columns else pool

    bad = ~out["label_str"].isin(label2id)
    if bad.any():
        print(f"[{pool}] dropping {bad.sum()} bad labels")
        out = out[~bad]

    out["label_id"] = out["label_str"].map(label2id).astype(int)
    print(f"Loaded {pool}: {len(out)} rows from {path}")
    return out.reset_index(drop=True)


train_parts = [load_clean_csv(path, pool) for path, pool in TRAIN_SOURCES]
df_train = dedupe_train(pd.concat(train_parts, ignore_index=True))
df_val = load_clean_csv(f"{DRIVE_SHRISHTI_CLEAN}/val.csv", "shrishti_val")
ood_eval = {name: load_clean_csv(path, name) for path, name in OOD_SOURCES}
ood_eval["adv_concept_ood"] = load_adv_concept_csv(ADV_CONCEPT_PATH_CANDIDATES)

print("Train rows:", len(df_train))
print(df_train.groupby("pool")["label_str"].value_counts())
print("\nVal rows:", len(df_val), dict(df_val["label_str"].value_counts()))
for name, odf in ood_eval.items():
    print("OOD", name, len(odf), dict(odf["label_str"].value_counts()))

# Fit scaler on train only. Transform val/OOD with the train scaler.
Xstat_train_raw, STATIC_FEATURE_NAMES = compute_static_matrix(df_train, desc="static/train")
Xstat_val_raw, _ = compute_static_matrix(df_val, STATIC_FEATURE_NAMES, desc="static/val")
Xstat_ood_raw = {name: compute_static_matrix(odf, STATIC_FEATURE_NAMES, desc=f"static/{name}")[0] for name, odf in ood_eval.items()}

static_scaler = StandardScaler()
Xstat_train = static_scaler.fit_transform(Xstat_train_raw).astype("float32")
Xstat_val = static_scaler.transform(Xstat_val_raw).astype("float32")
Xstat_ood = {name: static_scaler.transform(x).astype("float32") for name, x in Xstat_ood_raw.items()}
STATIC_DIM = Xstat_train.shape[1]
print("Static dim:", STATIC_DIM)

domain2id = {s: i for i, s in enumerate(sorted(set(pd.concat([df_train[SOURCE_COL], df_val[SOURCE_COL]]).astype(str))))}
num_domains = len(domain2id)
print("Domains:", domain2id)

df_train["domain_id"] = df_train[SOURCE_COL].map(domain2id).astype(int)
df_val["domain_id"] = df_val[SOURCE_COL].map(domain2id).astype(int)

manifest = {
    "train_pools": {str(k): int(v) for k, v in df_train["pool"].value_counts().items()},
    "ood": {str(k): int(len(v)) for k, v in ood_eval.items()},
    "static_dim": int(STATIC_DIM),
    "domain2id": domain2id,
}
with open(os.path.join(DRIVE_OUT_DIR, "data_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
joblib.dump({"scaler": static_scaler, "feature_names": STATIC_FEATURE_NAMES}, os.path.join(DRIVE_OUT_DIR, "static_scaler.joblib"))

In [ ]:
# Model definition: ELECTRA pooled embedding + static features + DANN
class ScalarMix(nn.Module):
    def __init__(self, mixture_size: int, trainable: bool = True) -> None:
        super().__init__()
        self.scalar_parameters = ParameterList([Parameter(torch.zeros(1), requires_grad=trainable) for _ in range(mixture_size)])  #scalar parameters , intially 0
        self.gamma = Parameter(torch.ones(1), requires_grad=trainable) #learning rate to tell how much imprtance should final representation be given

    def forward(self, tensors: List[torch.Tensor]) -> torch.Tensor:
        w = torch.nn.functional.softmax(torch.cat([p for p in self.scalar_parameters]), dim=0) #take the scalar , concatenate to one vector, then softmax it
        w = torch.split(w, 1)
        return self.gamma * sum(weight * t for weight, t in zip(w, tensors))


def grl_lambda_schedule(progress: float) -> float: #according to 
    progress = float(min(1.0, max(0.0, progress)))
    return GRL_LAMBDA_MAX * (2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)


class GradientReversalFunction(torch.autograd.Function): #manually definind how .backward() or .forward() should behave
    @staticmethod
    def forward(ctx: Any, x: Tensor, lambda_: float) -> Tensor: #send it as it is
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> Tuple[Tensor, None]: #reverse the gradoutput so that it can unlearn domain specific features
        return -ctx.lambda_ * grad_output, None


def apply_gradient_reversal(x: Tensor, lambda_: float) -> Tensor:
    return GradientReversalFunction.apply(x, float(lambda_))


class MLPHead(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.2) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, out_dim))

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class ElectraStaticDANN(nn.Module):
    def __init__(self, model_name: str, static_dim: int, num_classes: int, num_domains: int, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = int(self.encoder.config.hidden_size)
        n_layers = int(self.encoder.config.num_hidden_layers) + 1
        self.scalar_mix = ScalarMix(n_layers)
        self.dropout = nn.Dropout(dropout)
        fused_dim = hidden + int(static_dim)
        self.difficulty_head = MLPHead(fused_dim, num_classes, dropout)
        self.domain_head = MLPHead(fused_dim, num_domains, dropout)

    def encode_pooled(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        mixed = self.dropout(self.scalar_mix(list(out.hidden_states)))
        mask = attention_mask.unsqueeze(-1).float()
        return (mixed * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def fused(self, input_ids: Tensor, attention_mask: Tensor, static_features: Tensor) -> Tensor:
        pooled = self.encode_pooled(input_ids, attention_mask)
        return torch.cat([pooled, static_features.float()], dim=1)

    def forward(self, input_ids: Tensor, attention_mask: Tensor, static_features: Tensor, grl_lambda: float) -> Tuple[Tensor, Tensor]:
        fused = self.fused(input_ids, attention_mask, static_features)
        diff_logits = self.difficulty_head(fused)
        dom_logits = self.domain_head(apply_gradient_reversal(fused, grl_lambda))
        return diff_logits, dom_logits

    def difficulty_logits_only(self, input_ids: Tensor, attention_mask: Tensor, static_features: Tensor) -> Tensor:
        return self.difficulty_head(self.fused(input_ids, attention_mask, static_features))

    def freeze_encoder(self) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = True

    def freeze_encoder_except_top(self, n_layers: int) -> None:
        self.freeze_encoder()
        if hasattr(self.encoder, "encoder") and hasattr(self.encoder.encoder, "layer"):
            for layer in self.encoder.encoder.layer[-n_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True
        for p in self.scalar_mix.parameters():
            p.requires_grad = True


def combined_loss(diff_logits, dom_logits, labels, domains):
    ce_task = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ce_dom = nn.CrossEntropyLoss()
    diff_loss = ce_task(diff_logits, labels)
    dom_loss = ce_dom(dom_logits, domains)
    return diff_loss + DOMAIN_LOSS_ALPHA * dom_loss, diff_loss, dom_loss

print("Model class loaded")

In [ ]:
# Tokenizer and loaders
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextStaticDANNDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray, domains: np.ndarray, static_features: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)
        self.domains = domains.astype(int)
        self.static_features = static_features.astype("float32")

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(self.texts[idx], truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "static": torch.tensor(self.static_features[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "domain": torch.tensor(self.domains[idx], dtype=torch.long),
        }


def df_to_dann_loader(df: pd.DataFrame, static_features: np.ndarray, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextStaticDANNDataset(df[TEXT_COL].tolist(), df["label_id"].values, df["domain_id"].values, static_features),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
    )


train_loader = df_to_dann_loader(df_train, Xstat_train, shuffle=True)
val_loader = df_to_dann_loader(df_val, Xstat_val, shuffle=False)
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# Train: phase 1 frozen encoder, phase 2 DANN partial unfreeze
model = ElectraStaticDANN(MODEL_NAME, static_dim=STATIC_DIM, num_classes=3, num_domains=num_domains).to(device)
best_path = os.path.join(DRIVE_OUT_DIR, "best_model.pt")
phase2_final_path = os.path.join(DRIVE_OUT_DIR, "phase2_final.pt")
history: List[Dict] = []
best_val_f1 = -1.0
global_step = 0
total_steps = (PHASE1_EPOCHS + PHASE2_EPOCHS) * len(train_loader)


@torch.no_grad()
def evaluate_val() -> Dict[str, float]:
    model.eval()
    ys, preds = [], []
    for batch in val_loader:
        logits = model.difficulty_logits_only(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["static"].to(device),
        )
        ys.extend(batch["label"].numpy().tolist())
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return {
        "val_macro_f1": float(f1_score(ys, preds, labels=EVAL_LABELS, average="macro", zero_division=0)),
        "val_acc": float(accuracy_score(ys, preds)),
    }


def run_phase(phase_name: str, epochs: int, lr: float, grl_on: bool) -> None:
    global global_step, best_val_f1
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=min(WARMUP_STEPS, max(1, len(train_loader))),
        num_training_steps=max(1, epochs * len(train_loader)),
    )
    for epoch in range(epochs):
        model.train()
        run_diff, run_dom, n_batches = 0.0, 0.0, 0
        grl_l = 0.0
        for batch in tqdm(train_loader, desc=f"{phase_name} ep{epoch+1}/{epochs}"):
            progress = global_step / max(total_steps - 1, 1)
            grl_l = grl_lambda_schedule(progress) if grl_on else 0.0
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            stat = batch["static"].to(device)
            labels = batch["label"].to(device)
            domains = batch["domain"].to(device)
            diff_logits, dom_logits = model(ids, mask, stat, grl_lambda=grl_l)
            total, diff_loss, dom_loss = combined_loss(diff_logits, dom_logits, labels, domains)
            optimizer.zero_grad()
            total.backward()
            optimizer.step()
            scheduler.step()
            global_step += 1
            run_diff += diff_loss.item()
            run_dom += dom_loss.item()
            n_batches += 1
        metrics = evaluate_val()
        rec = {
            "phase": phase_name,
            "epoch": epoch + 1,
            "grl_on": grl_on,
            "grl_lambda": float(grl_l),
            "train_diff_loss": run_diff / max(n_batches, 1),
            "train_dom_loss": run_dom / max(n_batches, 1),
            **metrics,
        }
        history.append(rec)
        print(f"{phase_name} ep{epoch+1}: diff={rec['train_diff_loss']:.4f} dom={rec['train_dom_loss']:.4f} val_f1={rec['val_macro_f1']:.4f}")
        if metrics["val_macro_f1"] > best_val_f1:
            best_val_f1 = metrics["val_macro_f1"]
            torch.save({
                "state_dict": model.state_dict(),
                "domain2id": domain2id,
                "static_dim": STATIC_DIM,
                "static_feature_names": STATIC_FEATURE_NAMES,
            }, best_path)
            print(f"  saved best val F1={best_val_f1:.4f}")


model.freeze_encoder()
run_phase("phase1_frozen_encoder", PHASE1_EPOCHS, PHASE1_LR, grl_on=False)

model.unfreeze_encoder()
model.freeze_encoder_except_top(PARTIAL_FREEZE_LAYERS)
run_phase("phase2_dann", PHASE2_EPOCHS, PHASE2_LR, grl_on=True)

torch.save({
    "state_dict": model.state_dict(),
    "domain2id": domain2id,
    "phase": "phase2_dann_final",
    "static_dim": STATIC_DIM,
    "static_feature_names": STATIC_FEATURE_NAMES,
}, phase2_final_path)
print(f"Saved phase2 final: {phase2_final_path}")
print(f"Best val F1={best_val_f1:.4f}")

In [ ]:
# Eval both checkpoints on OOD corpora
def save_confusion_matrix(y_true, y_pred, title, out_path, labels):
    names = [id2label[i] for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True (judge)")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def load_checkpoint(path: str) -> None:
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])


@torch.no_grad()
def predict_texts(texts: List[str], static_features: np.ndarray, batch_size: int = 16) -> np.ndarray:
    model.eval()
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="predict", leave=False):
        enc = tokenizer(texts[i : i + batch_size], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        stat = torch.tensor(static_features[i : i + batch_size], dtype=torch.float32)
        logits = model.difficulty_logits_only(enc["input_ids"].to(device), enc["attention_mask"].to(device), stat.to(device))
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


def eval_split(name: str, y_true: np.ndarray, y_pred: np.ndarray, tag: str) -> Dict:
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    acc = accuracy_score(y_true, y_pred)
    f1_all = f1_score(y_true, y_pred, labels=EVAL_LABELS, average="macro", zero_division=0)
    f1_pres = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    print(f"\n{'='*60}\n[{tag}] {name}\n{'='*60}")
    print(classification_report(y_true, y_pred, labels=EVAL_LABELS, target_names=[id2label[i] for i in EVAL_LABELS], zero_division=0))
    print(f"Accuracy: {acc:.4f} | Macro-F1: {f1_all:.4f} | present-only: {f1_pres:.4f}")
    cm_path = os.path.join(CM_DIR, f"cm_{tag}_{name}.png")
    save_confusion_matrix(y_true, y_pred, f"{tag} - {name}", cm_path, EVAL_LABELS)
    return {"checkpoint": tag, "corpus": name, "n": int(len(y_true)), "accuracy": float(acc), "macro_f1_3class": float(f1_all), "macro_f1_present": float(f1_pres), "gold_classes": present}


CHECKPOINTS = [("phase1_best_val", best_path)]
if os.path.exists(phase2_final_path):
    CHECKPOINTS.append(("phase2_dann_final", phase2_final_path))

all_rows = []
for tag, path in CHECKPOINTS:
    print(f"\n{'#'*60}\nOOD eval: {tag}\n{'#'*60}")
    load_checkpoint(path)
    for name, odf in ood_eval.items():
        y_true = odf["label_id"].values
        y_pred = predict_texts(odf[TEXT_COL].tolist(), Xstat_ood[name])
        all_rows.append(eval_split(name, y_true, y_pred, tag))

comparison = pd.DataFrame(all_rows)
pivot = comparison.pivot(index="corpus", columns="checkpoint", values="macro_f1_3class")
print("\nOOD macro-F1:\n", pivot.to_string())
print("\nMean OOD macro-F1:\n", comparison.groupby("checkpoint")["macro_f1_3class"].mean().to_string())

comparison.to_csv(os.path.join(DRIVE_OUT_DIR, "eval_comparison.csv"), index=False)
pivot.reset_index().to_csv(os.path.join(DRIVE_OUT_DIR, "eval_pivot_macro_f1.csv"), index=False)
with open(os.path.join(DRIVE_OUT_DIR, "eval_results.json"), "w", encoding="utf-8") as f:
    json.dump({"history": history, "evaluations": all_rows, "best_val_f1": float(best_val_f1)}, f, indent=2)
print("\nSaved to", DRIVE_OUT_DIR)